# Large-Scale Image Classification with Transfer Learning and Apache Spark

**Course**: SE363 - Big Data Processing & Machine Learning  
**Student Name**: Võ Minh Tiến 
**Student ID**: 24521784  
**Dataset Name**: Oxford-IIIT Pet Dataset  
**Dataset URL**: [https://www.robots.ox.ac.uk/~vgg/data/pets/](https://www.robots.ox.ac.uk/~vgg/data/pets/)  

---
### Overview
This notebook presents an end-to-end distributed image classification system:
1. **Dataset**: 7,349 color photos across 37 dog and cat breeds from Oxford-IIIT Pet.
2. **Transfer Learning**: Pretrained **ResNet-18** ImageNet backbone with a fine-tuned 37-class linear classifier.
3. **Apache Spark Deployment**: Distributed batch inference evaluated on **3,669 test images** using both:
   * **`pandas_udf`** (streaming scalar iterator pattern)
   * **`predict_batch_udf`** (native Spark ML vectorized batch inference)
4. **Evaluation**: Comprehensive benchmark comparing accuracy, macro F1, latency, and throughput.


In [1]:
import os
import time
from pathlib import Path
from typing import Iterator, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import IntegerType

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")


PyTorch Version: 2.13.0+cu130
CUDA Available:  False


## 1. Dataset Preparation & Exploration

We load the Oxford-IIIT Pet dataset from local disk, configure ImageNet preprocessing transforms, and partition into train (2,944 images), validation (736 images), and test (3,669 images).


In [2]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class OxfordPetDataset(Dataset):
    def __init__(self, root: str = "oxford-iiit-pet", split: str = "trainval", transform=None):
        self.root = Path(root)
        self.transform = transform
        self.images_dir = self.root / "images"
        self.ann_file = self.root / "annotations" / f"{split}.txt"

        self.samples: List[Tuple[str, int]] = []
        with open(self.ann_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts or parts[0].startswith("#"):
                    continue
                img_path = str(self.images_dir / f"{parts[0]}.jpg")
                class_id = int(parts[1]) - 1  # 0-indexed
                self.samples.append((img_path, class_id))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

# Create dataset splits
trainval_raw = OxfordPetDataset(root="oxford-iiit-pet", split="trainval")
test_dataset = OxfordPetDataset(root="oxford-iiit-pet", split="test", transform=eval_transform)

# 80/20 train/val split with fixed random seed
generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(trainval_raw), generator=generator).tolist()
split_point = int(len(trainval_raw) * 0.8)

train_base = OxfordPetDataset(root="oxford-iiit-pet", split="trainval", transform=train_transform)
val_base = OxfordPetDataset(root="oxford-iiit-pet", split="trainval", transform=eval_transform)

train_dataset = Subset(train_base, indices[:split_point])
val_dataset = Subset(val_base, indices[split_point:])

print(f"Dataset summary:")
print(f"  Training samples:   {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Test samples:       {len(test_dataset):,}")
print(f"  Total images:       {len(train_dataset) + len(val_dataset) + len(test_dataset):,} (37 classes)")


Dataset summary:
  Training samples:   2,944
  Validation samples: 736
  Test samples:       3,669
  Total images:       7,349 (37 classes)


## 2. Transfer Learning Model Architecture

We load ImageNet-pretrained **ResNet-18**, freeze all convolutional feature extraction layers (`requires_grad = False`), and replace the final fully-connected layer (`fc`) with a linear projection for the 37 pet classes.


In [3]:
def build_transfer_model(num_classes: int = 37, pretrained: bool = True) -> nn.Module:
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)

    # Freeze convolutional feature extractor
    for param in model.parameters():
        param.requires_grad = False

    # Replace classification head
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = build_transfer_model(num_classes=37)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ResNet-18 Architecture Summary:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,} (only custom classification head)")


ResNet-18 Architecture Summary:
  Total parameters:     11,195,493
  Trainable parameters: 18,981 (only custom classification head)


## 3. Training & TorchScript Model Export

We train the custom classification head for 3 epochs using Cross-Entropy Loss and Adam optimizer, save the best weights (`checkpoints/best_model.pth`), and trace/export the model to TorchScript (`checkpoints/model_scripted.pt`).


In [4]:
checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_model_path = checkpoint_dir / "best_model.pth"
scripted_model_path = checkpoint_dir / "model_scripted.pt"

# If checkpoint already exists, load it; otherwise train
if best_model_path.exists() and scripted_model_path.exists():
    print(f"Found existing trained checkpoint: {best_model_path}")
    print(f"Found existing TorchScript model: {scripted_model_path}")
else:
    print("Training transfer learning model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Use num_workers=0 to prevent multiprocessing pickling issues in interactive notebook environments
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    best_val_acc = 0.0
    for epoch in range(1, 4):
        t_start = time.time()
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            train_correct += (out.argmax(1) == lbls).sum().item()
            train_total += lbls.size(0)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                loss = criterion(out, lbls)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (out.argmax(1) == lbls).sum().item()
                val_total += lbls.size(0)

        val_acc = val_correct / val_total
        print(f"Epoch [{epoch}/3] ({time.time()-t_start:.1f}s) - "
              f"Train Acc: {train_correct/train_total*100:.2f}% | "
              f"Val Acc: {val_acc*100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)

    # Export TorchScript
    model.load_state_dict(torch.load(best_model_path, map_location="cpu"))
    model.eval().to("cpu")
    traced = torch.jit.trace(model, torch.randn(1, 3, 224, 224))
    traced.save(str(scripted_model_path))
    print(f"Exported TorchScript model to {scripted_model_path}")

print("Model preparation complete.")


Training transfer learning model...


Epoch [1/3] (86.3s) - Train Acc: 42.73% | Val Acc: 70.92%


Epoch [2/3] (86.6s) - Train Acc: 82.74% | Val Acc: 82.88%


Epoch [3/3] (83.9s) - Train Acc: 87.47% | Val Acc: 85.46%


Exported TorchScript model to checkpoints/model_scripted.pt
Model preparation complete.


## 4. Initialize Apache Spark Session & Test DataFrame

We spin up an Apache Spark 4.2.0 session configured with Apache Arrow acceleration and convert our 3,669 test samples into a distributed Spark DataFrame partitioned across 4 workers.


In [5]:
spark = (
    SparkSession.builder
    .appName("OxfordPet_Spark_Inference")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Create DataFrame with absolute paths and true integer labels
test_rows = [(str(Path(p).resolve()), lbl) for p, lbl in test_dataset.samples]
test_df = spark.createDataFrame(test_rows, ["image_path", "label"]).repartition(4).cache()

total_test_count = test_df.count()
print(f"Spark DataFrame created with {total_test_count:,} test images across 4 partitions.")
test_df.show(5, truncate=60)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tienvm/.local/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/23 21:22:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark DataFrame created with 3,669 test images across 4 partitions.


+------------------------------------------------------------+-----+
|                                                  image_path|label|
+------------------------------------------------------------+-----+
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/ImageCla...|    0|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/ImageCla...|    1|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/ImageCla...|    1|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/ImageCla...|    0|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/ImageCla...|    0|
+------------------------------------------------------------+-----+
only showing top 5 rows


## 5. Deployment Method 1: `pandas_udf` (Pandas Iterator Pattern)

We implement `predict_pandas_udf` using the Pandas Iterator pattern (`Iterator[pd.Series] -> Iterator[pd.Series]`). The TorchScript model is loaded **once per partition worker**, and images are streamed in batches to ensure bounded memory consumption.


In [6]:
abs_script_path = str(scripted_model_path.resolve())

def compute_spark_metrics(df, pred_col="prediction", label_col="label"):
    rows = df.select(pred_col, label_col).collect()
    preds = np.array([r[0] for r in rows])
    labels = np.array([r[1] for r in rows])
    acc = float(np.mean(preds == labels))

    # Macro F1
    classes = np.unique(labels)
    f1s = []
    for c in classes:
        tp = np.sum((preds == c) & (labels == c))
        fp = np.sum((preds == c) & (labels != c))
        fn = np.sum((preds != c) & (labels == c))
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        f1s.append(f1)
    return acc, float(np.mean(f1s))

@pandas_udf(IntegerType())
def pandas_predict(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
    device = torch.device("cpu")
    m = torch.jit.load(abs_script_path, map_location=device)
    m.eval()

    with torch.no_grad():
        for paths in batch_iter:
            tensors = [eval_transform(Image.open(p).convert("RGB")) for p in paths]
            batch_t = torch.stack(tensors).to(device)
            preds = m(batch_t).argmax(dim=1).cpu().numpy().astype(int)
            yield pd.Series(preds)

print("Running distributed inference via pandas_udf...")
t0 = time.time()
df_pandas = test_df.withColumn("prediction", pandas_predict(col("image_path"))).cache()
df_pandas.count()  # Trigger action
pandas_time = time.time() - t0
pandas_fps = total_test_count / pandas_time

pandas_acc, pandas_f1 = compute_spark_metrics(df_pandas)

print(f"pandas_udf Execution Time: {pandas_time:.2f} s")
print(f"pandas_udf Throughput:     {pandas_fps:.1f} images/s")
print(f"pandas_udf Accuracy:       {pandas_acc*100:.2f}%")
print(f"pandas_udf Macro F1:       {pandas_f1*100:.2f}%")


Running distributed inference via pandas_udf...


/home/tienvm/.local/lib/python3.14/site-packages/pyspark/sql/pandas/functions.py:777: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
/home/tienvm/.local/lib/python3.14/site-packages/pyspark/sql/pandas/typehints.py:60: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()



[Stage 11:==============>                                           (1 + 3) / 4]



pandas_udf Execution Time: 67.29 s
pandas_udf Throughput:     54.5 images/s
pandas_udf Accuracy:       83.43%
pandas_udf Macro F1:       83.19%


## 6. Deployment Method 2: `predict_batch_udf` (Spark ML Native Batch Inference)

We implement `predict_batch_udf` utilizing PySpark ML's native vectorized batch inference API. It accepts mini-batches of NumPy ndarrays, minimizes serialization overhead, and executes batch predictions directly.


In [7]:
def make_predict_fn():
    device = torch.device("cpu")
    m = torch.jit.load(abs_script_path, map_location=device)
    m.eval()

    def predict(paths: np.ndarray) -> np.ndarray:
        tensors = [eval_transform(Image.open(str(p)).convert("RGB")) for p in paths]
        with torch.no_grad():
            batch_t = torch.stack(tensors).to(device)
            preds = m(batch_t).argmax(dim=1).cpu().numpy().astype(np.int32)
        return preds

    return predict

predict_batch_fn = predict_batch_udf(
    make_predict_fn=make_predict_fn,
    return_type=IntegerType(),
    batch_size=32,
)

print("Running distributed inference via predict_batch_udf...")
t0 = time.time()
df_batch = test_df.withColumn("prediction", predict_batch_fn(col("image_path"))).cache()
df_batch.count()  # Trigger action
batch_time = time.time() - t0
batch_fps = total_test_count / batch_time

batch_acc, batch_f1 = compute_spark_metrics(df_batch)

print(f"predict_batch_udf Execution Time: {batch_time:.2f} s")
print(f"predict_batch_udf Throughput:     {batch_fps:.1f} images/s")
print(f"predict_batch_udf Accuracy:       {batch_acc*100:.2f}%")
print(f"predict_batch_udf Macro F1:       {batch_f1*100:.2f}%")


Running distributed inference via predict_batch_udf...



[Stage 20:==============>                                           (1 + 3) / 4]



predict_batch_udf Execution Time: 73.07 s
predict_batch_udf Throughput:     50.2 images/s
predict_batch_udf Accuracy:       83.43%
predict_batch_udf Macro F1:       83.19%


## 7. Comparative Performance & Evaluation Summary

We present the side-by-side benchmark table comparing both deployment approaches across accuracy, macro F1, latency, and throughput on the 3,669 test images.


In [8]:
print("=" * 68)
print(f"{'Deployment Method':<22} | {'Accuracy':<10} | {'Macro F1':<10} | {'Throughput (img/s)':<18}")
print("-" * 68)
print(f"{'pandas_udf':<22} | {pandas_acc*100:>8.2f}% | {pandas_f1*100:>8.2f}% | {pandas_fps:>18.1f}")
print(f"{'predict_batch_udf':<22} | {batch_acc*100:>8.2f}% | {batch_f1*100:>8.2f}% | {batch_fps:>18.1f}")
print("=" * 68)

speedup = ((batch_fps - pandas_fps) / pandas_fps) * 100
print(f"\nThroughput Improvement (predict_batch_udf vs pandas_udf): +{speedup:.2f}%")

print("\nSample Distributed Predictions from Spark DataFrame:")
df_batch.select("image_path", "label", "prediction").show(10, truncate=55)

# Clean up Spark Session
spark.stop()
print("SparkSession stopped successfully.")


Deployment Method      | Accuracy   | Macro F1   | Throughput (img/s)
--------------------------------------------------------------------
pandas_udf             |    83.43% |    83.19% |               54.5
predict_batch_udf      |    83.43% |    83.19% |               50.2

Throughput Improvement (predict_batch_udf vs pandas_udf): +-7.91%

Sample Distributed Predictions from Spark DataFrame:
+-------------------------------------------------------+-----+----------+
|                                             image_path|label|prediction|
+-------------------------------------------------------+-----+----------+
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/Ima...|    0|         0|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/Ima...|    1|         1|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/Ima...|    1|         1|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/Ima...|    0|         0|
|/run/media/tienvm/WORK/UIT_AUTUMN2026-2027/SE363/Ima...|    0|         0|
|/ru

SparkSession stopped successfully.
